# 04. 날짜 기반 split 확정과 시계열 모델 조사

마지막 2개 거래일을 고정 Test로 두고, 앞 7개 거래일에서 expanding walk-forward OOF로 모델과 학습 설정을 선택한다. Test는 모델·threshold 선택에 사용하지 않는다.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = (PROJECT_ROOT / '../../results/preprocessing').resolve()
VERSION = 'ohlc_60m_tp3pct_v1'
metadata = pd.read_parquet(OUTPUT_ROOT / f'{VERSION}_metadata.parquet')
random_labels = pd.read_parquet(OUTPUT_ROOT / 'ohlc_60m_tp3pct_random_entry_v1_metadata.parquet')
features = np.load(OUTPUT_ROOT / f'{VERSION}_features.npz')

assert metadata['sample_id'].is_unique
assert metadata['sample_id'].tolist() == random_labels['sample_id'].tolist()
assert len(metadata) == len(features['sequence']) == len(features['context'])
print('samples:', len(metadata))

samples: 64144


## 1. Outer Train/Test

날짜 경계를 사용하므로 인접 60봉 window가 Train과 Test에 섞이지 않는다. 원본 파일과 라벨도 세션을 넘지 않으므로 별도 분 단위 embargo는 필요하지 않다. 단, 7/16~17은 과거 개발 과정에서 이미 확인한 날짜이므로 최종 pristine 승인용이 아니라 고정 development holdout으로 기록한다.

In [2]:
TRAIN_SESSIONS = [
    'session_2026-07-07', 'session_2026-07-08', 'session_2026-07-09',
    'session_2026-07-10', 'session_2026-07-13', 'session_2026-07-14',
    'session_2026-07-15',
]
TEST_SESSIONS = ['session_2026-07-16', 'session_2026-07-17']
assert set(metadata['session']) == set(TRAIN_SESSIONS + TEST_SESSIONS)

split = metadata[['sample_id', 'session', 'symbol', 'decision_timestamp', 'entry_timestamp']].copy()
split['split'] = np.where(split['session'].isin(TRAIN_SESSIONS), 'train', 'test')
split['oof_fold'] = 0

split_summary = split.groupby(['split', 'session']).size().rename('samples').reset_index()
split_totals = split.groupby('split').size().rename('samples').to_frame()
split_totals['share'] = split_totals['samples'] / len(split)
display(split_totals.style.format({'share': '{:.2%}'}))
display(split_summary)

,samples,share
split,,
test,11097,17.30%
train,53047,82.70%


,split,session,samples
0,test,session_2026-07-16,5338
1,test,session_2026-07-17,5759
2,train,session_2026-07-07,3781
3,train,session_2026-07-08,9147
4,train,session_2026-07-09,9038
5,train,session_2026-07-10,9077
6,train,session_2026-07-13,7297
7,train,session_2026-07-14,6120
8,train,session_2026-07-15,8587


## 2. Train 내부 expanding walk-forward OOF

첫 3일을 최소 학습 이력으로 두고 다음 날짜 하나씩 예측한다. 각 fold의 scaler는 해당 fold의 fit 날짜에서만 계산한다. OOF 4일의 평균뿐 아니라 날짜별 최악 성능을 모델 선택에 사용한다.

In [3]:
MINIMUM_PRIOR_SESSIONS = 3
folds = []
for evaluation_index in range(MINIMUM_PRIOR_SESSIONS, len(TRAIN_SESSIONS)):
    fold_number = len(folds) + 1
    fit_sessions = TRAIN_SESSIONS[:evaluation_index]
    evaluation_session = TRAIN_SESSIONS[evaluation_index]
    split.loc[split['session'].eq(evaluation_session), 'oof_fold'] = fold_number
    folds.append({
        'fold': fold_number,
        'fit_sessions': fit_sessions,
        'evaluation_session': evaluation_session,
        'fit_samples': int(split['session'].isin(fit_sessions).sum()),
        'evaluation_samples': int(split['session'].eq(evaluation_session).sum()),
    })

fold_table = pd.DataFrame([{**fold, 'fit_sessions': ', '.join(fold['fit_sessions'])} for fold in folds])
display(fold_table)
print('OOF samples:', int(split['oof_fold'].gt(0).sum()))
print('Initial history-only samples:', int((split['split'].eq('train') & split['oof_fold'].eq(0)).sum()))

,fold,fit_sessions,evaluation_session,fit_samples,evaluation_samples
0,1,"session_2026-07-07, session_2026-07-08, sessio...",session_2026-07-10,21966,9077
1,2,"session_2026-07-07, session_2026-07-08, sessio...",session_2026-07-13,31043,7297
2,3,"session_2026-07-07, session_2026-07-08, sessio...",session_2026-07-14,38340,6120
3,4,"session_2026-07-07, session_2026-07-08, sessio...",session_2026-07-15,44460,8587


OOF samples: 31081
Initial history-only samples: 21966


In [4]:
assert split.loc[split['split'].eq('test'), 'oof_fold'].eq(0).all()
assert split.loc[split['split'].eq('train'), 'session'].isin(TRAIN_SESSIONS).all()
assert metadata.loc[split['split'].eq('train'), 'decision_timestamp'].max() < metadata.loc[split['split'].eq('test'), 'decision_timestamp'].min()
for fold in folds:
    assert max(fold['fit_sessions']) < fold['evaluation_session']
print('시간 순서와 fold 누수 검사 통과')

시간 순서와 fold 누수 검사 통과


## 3. 최신 시계열 모델 조사

논문의 SOTA는 서로 다른 공개 benchmark에서 나온 결과이므로 현재 주식 데이터의 승자를 의미하지 않는다. 동일 OOF에서 직접 비교해야 한다.

| 모델 | 근거 | 현재 데이터 적합성 | 결정 |
|---|---|---|---|
| [TimeMixer++ (ICLR 2025)](https://proceedings.iclr.cc/paper_files/paper/2025/file/2b187165e28fdfdc0ffb34d1bfff2b0c-Paper-Conference.pdf) | 분류를 포함한 8개 분석 task에서 SOTA 보고, multi-scale·multi-resolution encoder | 60봉의 1·3·5분 패턴과 세 target head를 함께 다룰 수 있으나 복잡도가 큼 | **SOTA 주 후보** |
| [ModernTCN (ICLR 2024)](https://proceedings.iclr.cc/paper_files/paper/2024/file/86b1437c1e4c3b3c4debff98234a67e7-Paper-Conference.pdf) | 일반 시계열 분석용 순수 convolution, 효율성과 넓은 receptive field | 짧은 60봉·국소 캔들 패턴에 적합하고 multi-head 구현이 단순 | **필수 compact baseline** |
| [ModernTCN 재현성 감사 (TMLR 2025)](https://openreview.net/forum?id=R20kKdWmVZ) | 실험 설정을 교정하면 원 논문의 우위가 설정에 민감하다고 보고 | SOTA 명칭보다 동일 split 재평가가 중요함을 확인 | 결과 해석에 반영 |
| [MultiRocket](https://arxiv.org/abs/2102.00457) | 강한 TSC 정확도와 빠른 학습 | hard +3% 분류 sanity check에 유용하나 soft label·MFE/MAE multi-task에는 부적합 | **분류 reference** |
| [Mantis](https://arxiv.org/abs/2502.15637) | 사전학습 기반 시계열 분류와 multivariate adapter | 60봉을 사전학습 길이 512로 보간해야 하며 회귀 head가 기본 목적이 아님 | 1차 실험 제외 |
| [iTransformer (ICLR 2024)](https://openreview.net/forum?id=JePfAI8fah) | 변수 간 상관을 variate token attention으로 학습 | 원 논문의 주 목적이 장기 forecasting이라 현재 event 분류와 차이 큼 | 후순위 |

## 4. 모델 실험 결정

첫 비교는 동일한 날짜 fold, scaling, sampling, parameter budget을 사용한다.

1. `Compact ModernTCN`: 파이프라인과 과적합 여부를 확인하는 기준 모델
2. `Small TimeMixer++`: SOTA 구조를 현재 60봉 길이에 맞게 축소한 주 후보
3. `MultiRocket`: `t+1 open` hard 3분 +3% 분류 reference

ModernTCN과 TimeMixer++에는 같은 세 종류의 head를 각각 비교한다.

- Head A: `t+1 open` 기준 3분 +3% hard BCE
- Head A′: 랜덤 진입 3분 soft probability BCE
- Head C: 3분 MFE·MAE 회귀

1분·5분 target은 auxiliary로만 사용한다. 모델 선택은 OOF 4일의 PR-AUC, Spearman, 날짜별 최악 성능으로 하고 Test는 마지막 한 번만 계산한다.

## 5. Split artifact 저장

In [5]:
paths = {
    'split': OUTPUT_ROOT / f'{VERSION}_split.parquet',
    'summary': OUTPUT_ROOT / f'{VERSION}_split_summary.parquet',
    'folds': OUTPUT_ROOT / f'{VERSION}_walk_forward_folds.json',
}
split.to_parquet(paths['split'], index=False, compression='zstd')
split_summary.to_parquet(paths['summary'], index=False, compression='zstd')
fold_document = {
    'version': VERSION,
    'train_sessions': TRAIN_SESSIONS,
    'test_sessions': TEST_SESSIONS,
    'test_role': 'fixed development holdout',
    'test_is_pristine': False,
    'minimum_prior_sessions': MINIMUM_PRIOR_SESSIONS,
    'folds': folds,
    'scaler_rule': 'fit on fold fit_sessions only',
    'model_selection_data': 'OOF only',
}
paths['folds'].write_text(json.dumps(fold_document, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame([{'artifact': name, 'path': str(path), 'size_mb': path.stat().st_size / 1024**2} for name, path in paths.items()]))

,artifact,path,size_mb
0,split,/home/user/urbandatalab/YSLee/results/preproce...,0.409068
1,summary,/home/user/urbandatalab/YSLee/results/preproce...,0.002246
2,folds,/home/user/urbandatalab/YSLee/results/preproce...,0.001681


## 확정 사항

Train은 2026-07-07~15의 7개 세션, 고정 Test는 2026-07-16~17의 2개 세션이다. Train 내부 4개 expanding OOF 날짜로 모델을 선택한다. 다음 단계에서 Compact ModernTCN부터 학습 파이프라인을 검증한 뒤 Small TimeMixer++를 같은 조건으로 비교한다.